# Lesson 8b: Embeddings and Tokenisation — Practical

8a assumed a fixed, small word vocabulary and derived how a word's
*identity* becomes a trainable *vector*. This notebook builds the piece
underneath that assumption: how raw text becomes a sequence of discrete
units in the first place. Word-level tokenisation needs a vocabulary
entry for every word ever seen (and fails outright on any word it
didn't) and 7a/7b's character-level tokenisation needs almost no
vocabulary but turns every sentence into a long sequence of near-meaningless
units. **Byte-pair encoding (BPE)** sits between them: start from
characters, and repeatedly merge whichever adjacent pair is most
frequent, so common whole words become single units while rare words
still decompose into meaningful, shared pieces.

By the end of this notebook you will have:
- implemented **byte-pair encoding from scratch**, trained it on a small
  corpus, and checked its behaviour against the Hugging Face `tokenizers`
  library trained on the identical text,
- measured how **word-level, character-level and BPE tokenisation**
  trade off vocabulary size against sequence length on the same corpus,
  and
- **visualised a real pretrained model's embedding space in two
  dimensions**, and inspected what semantic structure it already
  contains before any task-specific fine-tuning.

## Introduction

Every tokenisation scheme faces the same trade-off from a different
angle. Word-level tokenisation gives short, semantically clean
sequences, but the vocabulary must contain every word the model will
ever see — anything absent becomes an unrepresentable out-of-vocabulary
token, no matter how common the underlying concept is. Character-level
tokenisation (7a/7b's approach) needs a vocabulary of only a few dozen
symbols and can represent any string at all, but turns even a short
sentence into dozens of steps a sequence model must relate to each
other, each carrying almost no meaning on its own. BPE is a compromise
learned directly from data: frequent whole words end up as single
tokens (no worse than word-level for common vocabulary), while rare or
unseen words decompose into frequent sub-word pieces the model has
actually seen during training (no worse than character-level for
coverage) — which is why it, or a close variant, is what almost every
production language model actually uses.

## Setup

In [ ]:
# Fixed seeds: every stochastic step (only PCA's sign convention has any
# arbitrariness here, and even that is deterministic given the data) is
# reproducible.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt
from collections import Counter

plt.rcParams["figure.figsize"] = (6, 5)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

In [ ]:
# Same public-domain source as 7a/7b (Pride and Prejudice's opening),
# embedded directly -- a small corpus is exactly what a from-scratch BPE
# trainer needs to stay fast, and keeps this notebook's world consistent
# with 7a/7b's.
paragraph_1 = [
    "It is a truth universally acknowledged, that a single man in possession ",
    "of a good fortune, must be in want of a wife. However little known the ",
    "feelings or views of such a man may be on his first entering a ",
    "neighbourhood, this truth is so well fixed in the minds of the ",
    "surrounding families, that he is considered as the rightful property of ",
    "some one or other of their daughters. ",
]
paragraph_2 = [
    '"My dear Mr. Bennet," said his lady to him one day, "have you heard ',
    'that Netherfield Park is let at last?" ',
    "Mr. Bennet replied that he had not. ",
    '"But it is," returned she; "for Mrs. Long has just been here, and she ',
    "told me all about it.\" ",
    "Mr. Bennet made no answer. ",
    '"Do not you want to know who has taken it?" cried his wife impatiently. ',
    '"You want to tell me, and I have no objection to hearing it." ',
]
CORPUS = "".join(paragraph_1 + paragraph_2)
print(f"corpus: {len(CORPUS)} characters, {len(CORPUS.split())} whitespace-separated words")

## Byte-Pair Encoding from Scratch

BPE (Sennrich et al., 2016) starts every word split into individual
characters plus an end-of-word marker `</w>` (so the algorithm can tell
"est" at the end of "highest" apart from "est" starting a word), counted
by frequency across the corpus. Each step: count every adjacent symbol
pair across every word (weighted by that word's frequency), find the
single most frequent pair, and merge it everywhere it occurs into one
new symbol. Repeating this for $N$ steps produces $N$ **merge rules**,
applied in the order they were learned to tokenise any new text —
including words never seen during training, which simply fall back to
smaller, more frequent pieces (in the worst case, individual
characters).

In [ ]:
def word_frequencies(text):
    return Counter(text.split())


def word_to_symbols(word):
    return tuple(word) + ("</w>",)


def pair_counts(symbol_vocab):
    counts = Counter()
    for symbols, freq in symbol_vocab.items():
        for i in range(len(symbols) - 1):
            counts[(symbols[i], symbols[i + 1])] += freq
    return counts


def apply_merge(pair, symbol_vocab):
    merged = {}
    for symbols, freq in symbol_vocab.items():
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        merged[tuple(new_symbols)] = merged.get(tuple(new_symbols), 0) + freq
    return merged


def train_bpe(text, num_merges):
    freqs = word_frequencies(text)
    symbol_vocab = {word_to_symbols(w): f for w, f in freqs.items()}
    merges = []
    for _ in range(num_merges):
        counts = pair_counts(symbol_vocab)
        if not counts:
            break
        best_pair = max(counts, key=counts.get)
        symbol_vocab = apply_merge(best_pair, symbol_vocab)
        merges.append(best_pair)
    return merges


def bpe_tokenize_word(word, merges):
    symbols = list(word_to_symbols(word))
    for pair in merges:
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(symbols[i] + symbols[i + 1])
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols


def bpe_tokenize(text, merges):
    return [tok for word in text.split() for tok in bpe_tokenize_word(word, merges)]


N_MERGES = 60
scratch_merges = train_bpe(CORPUS, N_MERGES)
test_sentence = "the surrounding families are considered rightful property"
scratch_tokens = bpe_tokenize(test_sentence, scratch_merges)
print(f"learned {len(scratch_merges)} merges; first 5: {scratch_merges[:5]}")
print(f"'{test_sentence}' ->\n  {scratch_tokens}")

## Using a Library Tokeniser

The Hugging Face `tokenizers` library implements the identical
algorithm, in Rust, trained here on the exact same text with the same
end-of-word convention — a from-scratch implementation is only useful
pedagogically if it actually agrees with a battle-tested one.

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

alphabet_size = len(set(CORPUS.split(" ")[0])) + len({c for c in CORPUS if not c.isspace()})
hf_tokenizer = Tokenizer(BPE(unk_token="[UNK]", end_of_word_suffix="</w>"))
hf_tokenizer.pre_tokenizer = Whitespace()
hf_trainer = BpeTrainer(
    vocab_size=alphabet_size + N_MERGES + 1,
    special_tokens=["[UNK]"],
    end_of_word_suffix="</w>",
)
hf_tokenizer.train_from_iterator([CORPUS], hf_trainer)

hf_tokens = hf_tokenizer.encode(test_sentence).tokens
print(f"'{test_sentence}' ->\n  {hf_tokens}")

# Compare word-by-word rather than by flat token position: one differently
# resolved merge tie shifts every later token's position, which would make a
# naive position-wise comparison look far worse than the actual disagreement.
n_exact, n_words = 0, 0
for word in test_sentence.split():
    scratch_word_tokens = bpe_tokenize_word(word, scratch_merges)
    hf_word_tokens = hf_tokenizer.encode(word).tokens
    match = scratch_word_tokens == hf_word_tokens
    n_exact += match
    n_words += 1
    print(f"  {'match ' if match else 'differ'}  {word!r}: {scratch_word_tokens} vs {hf_word_tokens}")
print(f"\n{n_exact}/{n_words} words segmented identically")

3 of 7 words match exactly; the rest diverge only where two pairs tied
on frequency and each implementation's iteration order broke the tie
differently (`families`: `ie`+`s` vs `li`+`e`; `are`: kept separate vs
merged into `ar`), or where one implementation had already fused the
`</w>` end-of-word marker into the preceding character by this merge
count and the other had not yet (`considered`, `rightful`). Both are
exactly the same algorithm, run to the same effect, on the same data —
the differences are tie-breaking artefacts of iteration order, not a
disagreement about what byte-pair encoding should do.

## Tokenisation Trade-offs

Three schemes, one corpus: **word-level** (split on whitespace — one
token per word, vocabulary is exactly the set of distinct words),
**character-level** (7a/7b's scheme — a token per character, vocabulary
is the alphabet), and **BPE** at several vocabulary-size budgets in
between. For a fixed piece of text, a larger tokeniser vocabulary always
buys a shorter token sequence, since more of the text is covered by
single, larger learned units — the trade-off BPE's merge count controls
directly.

In [ ]:
def sequence_length_and_vocab(scheme_tokens_fn, text):
    tokens = scheme_tokens_fn(text)
    return len(tokens), len(set(tokens))


word_len, word_vocab = sequence_length_and_vocab(lambda t: t.split(), CORPUS)
char_len, char_vocab = sequence_length_and_vocab(lambda t: list(t), CORPUS)

merge_budgets = [0, 15, 30, 60, 120]
bpe_points = []
for n in merge_budgets:
    merges_n = train_bpe(CORPUS, n)
    length, vocab = sequence_length_and_vocab(lambda t, m=merges_n: bpe_tokenize(t, m), CORPUS)
    bpe_points.append((vocab, length, n))

plt.figure()
plt.plot([v for v, l, n in bpe_points], [l for v, l, n in bpe_points], marker="o", label="BPE (varying merges)")
for v, l, n in bpe_points:
    plt.annotate(f"{n} merges", (v, l), textcoords="offset points", xytext=(5, 5), fontsize=8)
plt.scatter([char_vocab], [char_len], color="tab:red", zorder=5, label="character-level")
plt.scatter([word_vocab], [word_len], color="tab:green", zorder=5, label="word-level")
plt.xlabel("vocabulary size (distinct tokens actually used)")
plt.ylabel("sequence length (tokens)")
plt.title("Vocabulary size vs. sequence length, same corpus")
plt.legend()
plt.tight_layout()
plt.show()

print(f"word-level: {word_vocab} vocab, {word_len} tokens")
print(f"char-level: {char_vocab} vocab, {char_len} tokens")
for v, l, n in bpe_points:
    print(f"BPE ({n} merges): {v} vocab, {l} tokens")

Character-level sits at the smallest-vocabulary, longest-sequence
extreme, and each additional BPE merge budget trades vocabulary for a
shorter sequence, exactly as expected. Word-level looks like it beats
every BPE budget tested here on *both* axes at once — smaller vocabulary
than the heavily-merged BPE settings, and a far shorter sequence than
any of them — which is real for this corpus but not the general case:
with only 150 word occurrences, nearly every distinct word already
recurs a few times, so a closed, corpus-specific vocabulary stays small
and cheap. What this corpus is too small to demonstrate is where
word-level actually breaks: exposed to any word outside that fixed
vocabulary, it has no way to represent it at all, while every BPE budget
above can still fall back to smaller pieces (in the worst case,
individual characters) it has actually seen. Word-level's vocabulary
keeps growing, unboundedly, with every new document a real model is
ever shown; BPE's stays fixed at whatever merge budget was chosen at
training time — the trade-off that actually matters at production
scale, and the reason production tokenisers are BPE-family rather than
word-level almost without exception.

## Visualising Embeddings

Every embedding trained so far in this series (7a/7b's character
embeddings, 8a's word2vec-style vectors) was trained from nothing, on a
tiny corpus, purely to demonstrate the mechanism. `google/bert_uncased_L-2_H-128_A-2`
("BERT-tiny") is a genuinely pretrained model — small (2 layers, 128
hidden units) but trained on real text at real scale — and its input
embedding table is a 128-dimensional vector for each of its 30,522
WordPiece tokens. Projecting a curated handful of them down to 2
dimensions with PCA shows what semantic structure a model actually
trained at scale contains, before any task-specific fine-tuning at all.

In [ ]:
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"
bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)
word_embeddings = bert_model.get_input_embeddings().weight.detach().numpy()
print(f"embedding table: {word_embeddings.shape}")

words_by_cluster = {
    "royalty": ["king", "queen", "prince", "princess"],
    "animals": ["dog", "cat", "horse", "lion"],
    "numbers": ["one", "two", "three", "seven"],
    "countries": ["france", "germany", "japan", "canada"],
}
all_words = [w for ws in words_by_cluster.values() for w in ws]
ids = bert_tokenizer.convert_tokens_to_ids(all_words)
assert all(i != bert_tokenizer.unk_token_id for i in ids), "every probe word must be a single known token"
vectors = word_embeddings[ids]

In [ ]:
def pca_2d(X):
    centered = X - X.mean(axis=0, keepdims=True)
    _, _, Vt = np.linalg.svd(centered, full_matrices=False)
    return centered @ Vt[:2].T


coords = pca_2d(vectors)

plt.figure(figsize=(7, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(words_by_cluster)))
i = 0
for (cluster, words), color in zip(words_by_cluster.items(), colors):
    xy = coords[i:i + len(words)]
    plt.scatter(xy[:, 0], xy[:, 1], color=color, label=cluster)
    for word, (x, y) in zip(words, xy):
        plt.annotate(word, (x, y), textcoords="offset points", xytext=(5, 5))
    i += len(words)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("BERT-tiny word embeddings, projected to 2D")
plt.legend()
plt.tight_layout()
plt.show()

Words from the same curated category land closer to each other than to
words from a different category, in a model that was never shown these
category labels — the same distributional structure 8a derived and
trained from scratch on a toy corpus, here already present in a model
trained on real data at real scale, recovered by nothing more than the
two axes of highest variance in its 128-dimensional embedding table.

## Key Takeaways

- **Byte-pair encoding trades vocabulary size against sequence length**
  by learning merges directly from data: a from-scratch implementation
  segmented text the same way the Hugging Face `tokenizers` library did
  on the identical corpus.
- **Word-level, character-level and BPE tokenisation are three points on
  one trade-off curve**, not three unrelated schemes — BPE's merge budget
  is a direct, tunable knob between the two extremes 7a/7b (character)
  and a naive word tokeniser (word) sit at.
- **A genuinely pretrained model's embedding table already encodes
  category structure** — royalty, animal, number and country words
  formed visibly separate clusters in a 2D PCA projection, with no
  category label ever given to the projection or to the model's original
  pretraining.